# Expert Profiling in Qwen3.6-35B-A3B — Phase 1 of Paper 2

**Goal**: for each expert × each layer, measure how preferentially it activates on **correct** vs **wrong** rollouts. This gives us per-expert discriminative power for reasoning. Experts with high positive discriminative score are candidates for router-bias amplification in Phase 2.

**Protocol**:
1. Load Qwen3.6-35B-A3B (128 experts, 8 active per token, 40 layers)
2. Hook the `.mlp.gate` (router Linear) at every layer to capture router logits
3. For each Stage B rollout (711 total), replay prompt + first-50 response tokens → forward → capture which experts were top-8 at each (layer, token) and compute per-rollout activation frequency per expert
4. Aggregate: `mean_rate_correct - mean_rate_wrong` per (layer, expert) = discriminative power
5. Rank experts per layer, save results

**Budget**: ~1.5h compute on RTX 6000 Blackwell 96GB (1 forward per rollout at ~5s).

**Deliverable**: `expert_discriminative.pkl` — shape `[40 layers, 128 experts]` — plus top-10 helpers and top-10 anti per layer.

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path

def pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception:
    ok = False

if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'matplotlib', 'hf_transfer', check=False)
    pip('uninstall', '-y', 'transformers', 'causal-conv1d', check=False)
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC, check=False)
    pip('install','--no-cache-dir','flash-linear-attention', check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'):
            del sys.modules[m]

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('HF auth OK')
except Exception as e:
    print(f'(skipping: {e})')

import json, re, time, random, pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/expert_profiling'); OUT.mkdir(exist_ok=True)
print('setup done')

## Cell 2 — Load Qwen3.6-35B-A3B

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
from huggingface_hub import snapshot_download
from safetensors import safe_open

MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
model.eval()
print(f'Model loaded: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Inspect structure to find routers
def get_all_layers(m):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers')]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                return cur
    raise RuntimeError('layers not found')

all_layers = get_all_layers(model)
N_LAYERS = len(all_layers)
print(f'Total layers: {N_LAYERS}')

# Identify which layers have MoE (have .mlp.gate) — exclude GDN layers
moe_layers = []
for i, layer in enumerate(all_layers):
    if hasattr(layer, 'mlp') and hasattr(layer.mlp, 'gate'):
        moe_layers.append(i)
print(f'MoE layers (have .mlp.gate): {len(moe_layers)}')
print(f'Sample MoE layer indices: {moe_layers[:15]}...')

# Router dimension check
sample_gate = all_layers[moe_layers[0]].mlp.gate
print(f'Router Linear: in={sample_gate.in_features}, out={sample_gate.out_features} (= n_experts)')
N_EXPERTS = sample_gate.out_features

## Cell 3 — Install router hooks

Hook each MoE layer's `.mlp.gate` Linear module. Capture router logits per forward. Compute top-8 expert mask → per-expert activation frequency across tokens.

In [ ]:
TOP_K = 8  # Qwen3.6: 8 active per token

# Storage per rollout: {layer_idx: [n_experts] activation freq}
captured_freq = {}
captured_mask_sum = {}  # alternative: total count across tokens for double-checking
_current_num_tokens = {'val': 0}

def make_router_hook(layer_idx):
    def hook(module, inp, out):
        # out: [B, T, n_experts] router logits
        # Top-k (indices only)
        with torch.no_grad():
            top_k_idx = out.topk(TOP_K, dim=-1).indices  # [B, T, TOP_K]
            mask = torch.zeros_like(out, dtype=torch.bool)
            mask.scatter_(-1, top_k_idx, True)  # [B, T, n_experts]
            freq = mask.float().mean(dim=(0, 1))  # [n_experts] — fraction of tokens where this expert was active
            captured_freq[layer_idx] = freq.cpu().numpy()
    return hook

handles = []
for L in moe_layers:
    h = all_layers[L].mlp.gate.register_forward_hook(make_router_hook(L))
    handles.append(h)
print(f'✅ {len(handles)} router hooks installed')

## Cell 4 — Load Stage B rollouts (with response tokens + correctness)

In [ ]:
corpus = snapshot_download('caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
                            repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

MIN_LEN = 200
POOL_WINDOW = 50  # first 50 response tokens (longer window for routing stats)

rollouts = []
for shard in tqdm(shards, desc='load rollouts'):
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q_options = json.loads(meta['options'])
        question = meta['question']
        rts = json.loads(meta['response_tokens'])
        rs = json.loads(meta['rollouts'])
        for r_idx, r in enumerate(rs):
            if r['pred'] is None or r['response_len'] < MIN_LEN: continue
            rollouts.append(dict(
                question=question, options=q_options,
                gold_letter=meta['gold_letter'], pred=r['pred'],
                correct=r['correct'],
                response_tokens=rts[r_idx][:POOL_WINDOW]))

print(f'{len(rollouts)} rollouts loaded')
n_correct = sum(1 for r in rollouts if r['correct'])
print(f'Correct: {n_correct} | Wrong: {len(rollouts)-n_correct}')
print(f'Accuracy: {n_correct/len(rollouts)*100:.1f}%')

def format_mcq(question, options):
    choices = '\n'.join(f'{chr(65+i)}. {opt}' for i, opt in enumerate(options))
    content = (
        "Answer the following multiple-choice question. Reason step by step, "
        "then put the letter of the correct answer in \\boxed{}.\n\n"
        f"Question: {question}\n\nOptions:\n{choices}")
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True)

## Cell 5 — Profile: run 1 forward per rollout, capture router decisions

~1.5h on RTX 6000 (5s/rollout × 711).

In [ ]:
# Storage: [n_rollouts, n_moe_layers, n_experts]
expert_freq = np.zeros((len(rollouts), len(moe_layers), N_EXPERTS), dtype=np.float32)
rollout_correct = np.zeros(len(rollouts), dtype=bool)

t0 = time.time()
for ri, r in enumerate(tqdm(rollouts, desc='profile')):
    try:
        p = format_mcq(r['question'], r['options'])
        prompt_ids = tok(p, return_tensors='pt').input_ids.to('cuda')
        resp_ids = torch.tensor([r['response_tokens']], device='cuda')
        full_ids = torch.cat([prompt_ids, resp_ids], dim=1)
        if full_ids.shape[1] > 4096: continue

        with torch.no_grad():
            _ = model(input_ids=full_ids, attention_mask=torch.ones_like(full_ids))

        # captured_freq is now filled per layer
        for li, L in enumerate(moe_layers):
            expert_freq[ri, li, :] = captured_freq[L]
        rollout_correct[ri] = r['correct']

        if (ri + 1) % 50 == 0:
            elapsed = (time.time() - t0) / 60
            eta = elapsed / (ri + 1) * (len(rollouts) - ri - 1)
            print(f'  {ri+1}/{len(rollouts)} | elapsed {elapsed:.1f} min | ETA {eta:.1f} min')
            # Save incremental
            np.savez(OUT / 'expert_freq_partial.npz',
                     expert_freq=expert_freq[:ri+1],
                     correct=rollout_correct[:ri+1],
                     moe_layers=np.array(moe_layers))
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache(); continue
    except Exception as e:
        print(f'rollout {ri}: {e}'); continue

# Final save
np.savez(OUT / 'expert_freq.npz',
         expert_freq=expert_freq, correct=rollout_correct,
         moe_layers=np.array(moe_layers))
print(f'\n✅ Done in {(time.time()-t0)/60:.1f} min')
print(f'Saved to {OUT}/expert_freq.npz')
print(f'Shape: {expert_freq.shape} = [{len(rollouts)} rollouts, {len(moe_layers)} MoE layers, {N_EXPERTS} experts]')

## Cell 6 — Compute discriminative scores + rankings

In [ ]:
# Load if needed
data = np.load(OUT / 'expert_freq.npz')
expert_freq = data['expert_freq']
rollout_correct = data['correct']
moe_layers = list(data['moe_layers'])

correct_mask = rollout_correct.astype(bool)
print(f'Correct rollouts: {correct_mask.sum()} | Wrong: {(~correct_mask).sum()}')

# Per-layer-per-expert: mean activation rate on correct vs wrong
mean_correct = expert_freq[correct_mask].mean(axis=0)  # [n_moe_layers, n_experts]
mean_wrong = expert_freq[~correct_mask].mean(axis=0)
discriminative = mean_correct - mean_wrong  # positive = helper, negative = anti

# Effect size (Cohen's d) per expert
std_all = expert_freq.std(axis=0).clip(min=1e-6)
cohens_d = discriminative / std_all

print(f'\ndiscriminative shape: {discriminative.shape}')
print(f'Range: min {discriminative.min():.4f} | max {discriminative.max():.4f}')
print(f'Cohen\'s d: min {cohens_d.min():.3f} | max {cohens_d.max():.3f}')

# Save
np.savez(OUT / 'expert_discriminative.npz',
         discriminative=discriminative, cohens_d=cohens_d,
         mean_correct=mean_correct, mean_wrong=mean_wrong,
         moe_layers=np.array(moe_layers))

# Top-10 helpers + anti per layer for L15-L20 (our reasoning zone)
REASONING_LAYERS = [15, 16, 17, 18, 19, 20]
print('\n=== Reasoning zone (L15-L20) top experts ===')
for L in REASONING_LAYERS:
    if L not in moe_layers:
        print(f'L{L}: not MoE layer (probably GDN)'); continue
    li = moe_layers.index(L)
    disc_layer = discriminative[li]
    d_layer = cohens_d[li]
    top_helpers = np.argsort(-disc_layer)[:10]
    top_anti = np.argsort(disc_layer)[:10]
    print(f'\nL{L}:')
    print(f'  Helpers (high on correct):')
    for e in top_helpers[:5]:
        print(f'    expert {e:3d}: Δ{disc_layer[e]:+.4f} | d {d_layer[e]:+.3f} | rate_correct {mean_correct[li,e]:.3f} vs rate_wrong {mean_wrong[li,e]:.3f}')
    print(f'  Anti (high on wrong):')
    for e in top_anti[:5]:
        print(f'    expert {e:3d}: Δ{disc_layer[e]:+.4f} | d {d_layer[e]:+.3f} | rate_correct {mean_correct[li,e]:.3f} vs rate_wrong {mean_wrong[li,e]:.3f}')

print('\n✅ saved to expert_discriminative.npz')

## Cell 7 — Visualization + heatmap

In [ ]:
import matplotlib.pyplot as plt

# Max |discriminative| per layer — tells us which layers have most specialized experts
max_abs_per_layer = np.abs(discriminative).max(axis=1)
top_disc_per_layer = discriminative.max(axis=1)
bot_disc_per_layer = discriminative.min(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Per-layer max discriminative
axes[0].plot(moe_layers, top_disc_per_layer, 'g-', marker='o', label='best helper')
axes[0].plot(moe_layers, bot_disc_per_layer, 'r-', marker='o', label='worst anti')
axes[0].axvspan(15, 20, alpha=0.2, color='yellow', label='reasoning zone L15-L20')
axes[0].set_xlabel('Layer')
axes[0].set_ylabel('Activation rate difference (correct - wrong)')
axes[0].set_title('Per-layer expert specialization')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Cohen's d for top expert per layer
axes[1].plot(moe_layers, np.abs(cohens_d).max(axis=1), 'b-', marker='s')
axes[1].axvspan(15, 20, alpha=0.2, color='yellow', label='reasoning zone')
axes[1].axhline(0.5, linestyle='--', color='gray', label='d=0.5 (medium effect)')
axes[1].set_xlabel('Layer'); axes[1].set_ylabel("Max |Cohen's d|")
axes[1].set_title('Strongest expert effect per layer')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT / 'expert_discriminative_plot.png', dpi=120, bbox_inches='tight')
plt.show()

# Heatmap of reasoning zone layers only
fig, ax = plt.subplots(figsize=(20, 4))
reasoning_li = [moe_layers.index(L) for L in REASONING_LAYERS if L in moe_layers]
if reasoning_li:
    data_slice = discriminative[reasoning_li]
    im = ax.imshow(data_slice, aspect='auto', cmap='RdBu_r', vmin=-0.05, vmax=0.05)
    ax.set_yticks(range(len(reasoning_li)))
    ax.set_yticklabels([f'L{L}' for L in REASONING_LAYERS if L in moe_layers])
    ax.set_xlabel('Expert index (0-127)')
    ax.set_title('Discriminative rate (correct - wrong) per expert in reasoning zone')
    plt.colorbar(im, ax=ax, label='Δ rate')
    plt.tight_layout()
    plt.savefig(OUT / 'expert_heatmap_reasoning.png', dpi=120, bbox_inches='tight')
    plt.show()

# Summary stats
print('\n=== Summary ===')
max_disc_layer = moe_layers[np.argmax(np.abs(discriminative).max(axis=1))]
max_disc_val = np.abs(discriminative).max()
print(f'Max |Δ rate|: {max_disc_val:.4f} at L{max_disc_layer}')

n_strong_experts = (np.abs(cohens_d) > 0.5).sum()
print(f'Experts with |Cohen\'s d| > 0.5: {n_strong_experts}/{cohens_d.size}')
print(f'Experts with |Cohen\'s d| > 1.0: {(np.abs(cohens_d) > 1.0).sum()}/{cohens_d.size}')

# Per-reasoning-layer strong expert counts
print('\nReasoning-zone experts with strong discriminative power:')
for L in REASONING_LAYERS:
    if L not in moe_layers: continue
    li = moe_layers.index(L)
    n_strong = (np.abs(cohens_d[li]) > 0.5).sum()
    n_very_strong = (np.abs(cohens_d[li]) > 1.0).sum()
    print(f'  L{L}: {n_strong} experts |d|>0.5, {n_very_strong} |d|>1.0')
